# Weather summary

We'll try to get an advice on what to wear for the next week for a specific city.

## Helpers

In [ ]:
import json
from typing import Any
from IPython.display import DisplayHandle, Markdown, display


type Json = dict[str, Any]


def format_json(data: Json) -> str:
    """
    Format the json data as string.
    """
    return json.dumps(data, indent=2)


def display_markdown(markdown: str) -> DisplayHandle | None:
    return display(Markdown(markdown))

## open-meteo.com API stuff

In [ ]:
import requests
from typing import TypedDict


class Coordinates(TypedDict):
    latitude: float
    longitude: float


def get_city_coordinates(city: str) -> Coordinates | None:
    """
    Get the latitude and longitude of a city.
    """
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&language=en&format=json")
    response.raise_for_status()
    data = response.json()
    if "results" in data and (results := data["results"]):
        return results[0]
    return None


def get_weather_data(coordinates: Coordinates) -> Json:
    """
    Get the weather data for a specific city for the next week.
    """
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={coordinates['latitude']}&longitude={coordinates['longitude']}&daily=weathercode,temperature_2m_max,temperature_2m_min&timezone=auto")
    response.raise_for_status()
    return response.json()


## LLM interaction code

In [ ]:
from openai import OpenAI


weather_system_prompt = """
You are a assistant that analyzes weather in a specific city for the next week,
and provides a short summary of the weather provided to you in a JSON format.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

weather_user_prompt = """
Here are the weather data in a JSON format.
Provide a short summary of the weather in a markdown format.

{weather_data}
"""


advice_system_prompt = """
You are a assistant that analyzes weather summary for the next week and suggests a set of clothes for the next week.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

advice_user_prompt = """
Here are the weather summary for the next week.
Suggest a set of clothes for a week suitable for this weather summary.

{weather_summary}
"""


OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


def summarize_weather_data(weather_data: Json) -> str:
    messages = [
        {"role": "system", "content": weather_system_prompt},
        {"role": "user", "content": weather_user_prompt.format(weather_data=format_json(weather_data))}
    ]
    response = ollama.chat.completions.create(model="llama3.2:1b", messages=messages)
    return response.choices[0].message.content


def generate_clothes_advice(weather_summary: str) -> str:
    messages = [
        {"role": "system", "content": advice_system_prompt},
        {"role": "user", "content": advice_user_prompt.format(weather_summary=format_json(weather_summary))}
    ]
    response = ollama.chat.completions.create(model="llama3.2:1b", messages=messages)
    return response.choices[0].message.content


## Put it all together

In [ ]:
def get_clothes_advice(city: str) -> DisplayHandle | None:
    coordinates = get_city_coordinates(city)
    if coordinates is None:
        return display(f"Could not find coordinates for {city}")
    weather_data = get_weather_data(coordinates)
    summary = summarize_weather_data(weather_data)
    advice = generate_clothes_advice(summary)
    return display_markdown(advice)


## Demo

In [ ]:
get_clothes_advice("San Francisco")

In [ ]:
get_clothes_advice("London")

In [ ]:
get_clothes_advice("Nonexistingcity")